# Calculating the Allan Variance for our dataset.

This is ultimately the primary measurement for this particular study. Allan variance (AVAR) is the degree to which a signal varies over time. For the purposes of te current study, it's fairly simple to write it as

(1)   $AVAR = \sigma^2(\tau) = \frac{1}{2\tau^2} \left\langle (H[x|y_{\Delta_{t+2}}] - \big[ 2 \cdot H[x|y_{\Delta_{t+1}}] \big] - H[x|y_{\Delta_{t}}])^2 \right\rangle$

And we can test for various scaling relationships according to a regression of the log-transformed time $\Delta_t$ and the log-transformed $\sigma^2(\tau)$

(2)   $\log \sigma^2(\tau) \sim \alpha \log \Delta_t$

where $\alpha$ is the rate of exponential growth or decay.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(os.path.exists(FINAL_DOC))

True


In [3]:
# Allan Var doc
allan_var = 'allan_var.parquet'

#### Load data

In [4]:
main_df = pd.read_parquet(FINAL_DOC)
main_df.shape

(25424826, 13)

In [5]:
main_df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,1
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,2
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,3
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,4
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,5


#### Additional checks

In [6]:
main_df['conversation_id'].nunique()

2662

In [7]:
main_df['null'].value_counts()

null
False    23233133
True      2191693
Name: count, dtype: int64

In [8]:
main_df.loc[main_df['null']].head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta
23233133,8.396281,29.0,25.0,1,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,1
23233134,9.874767,29.0,5.0,3,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,2
23233135,8.867210,29.0,16.0,6,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,3
23233136,9.359125,29.0,5.0,8,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,4
23233137,8.732200,29.0,8.0,11,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,5


In [9]:
main_df.shape

(25424826, 13)

### Some specific checks to ensure that null data is not a replica of regular data

In [10]:
main_df['null'].sum(), main_df['null'].mean()

(np.int64(2191693), np.float64(0.08620287116222546))

In [11]:
tids = main_df['groups'].loc[main_df['null']].unique()
tids

array(['null-callfriend-eng_n-4175-19', 'null-callfriend-eng_n-4175-31',
       'null-callfriend-eng_n-4175-34', ...,
       'null-se3-ag4-se3-ag4-se3_ag4_m_02_5-43',
       'null-se3-ag4-se3-ag4-se3_ag4_m_02_5-46',
       'null-se3-ag4-se3-ag4-se3_ag4_m_02_5-51'], dtype=object)

In [12]:
main_df['groups'].loc[main_df['groups'].isin([tid.replace('null-', '') for tid in tids]) & (~main_df['null'])].nunique() / len(tids)


1.0

In [13]:
check_df = main_df.loc[main_df['groups'].isin(tids[:3])].copy()
check_df['groups'] = [tid.replace('null-', '') for tid in check_df['groups']]

check_df = pd.merge(
    left=check_df, left_on=['groups', 'self', 'sample_delta'],
    right=main_df[['groups', 'self', 'sample_delta', 'Hxy']].loc[~main_df['null']], right_on=['groups', 'self', 'sample_delta']
)
print(list(check_df))
check_df[['groups', 'Hxy_y', 'Hxy_x']].head(n=100)

['Hxy_x', 'nx', 'ny', 'turn_delta', 'self', 'x_speaker', 'dyad', 'corpus', 'conversation_id', 'x_turn_id', 'null', 'groups', 'sample_delta', 'Hxy_y']


,groups,Hxy_y,Hxy_x
0,callfriend-eng_n-4175-19,9.724691,8.396281
1,callfriend-eng_n-4175-19,9.718081,9.874767
2,callfriend-eng_n-4175-19,9.729723,8.867210
3,callfriend-eng_n-4175-19,7.685726,9.359125
4,callfriend-eng_n-4175-19,8.528924,8.732200
...,...,...,...
95,callfriend-eng_n-4175-34,1.104928,1.002753
96,callfriend-eng_n-4175-34,0.714613,1.099127
97,callfriend-eng_n-4175-34,1.094481,1.085704
98,callfriend-eng_n-4175-34,1.062234,1.095373


## Merge `copied doc` Method.

### Regular Data

In [6]:
df = main_df.loc[~main_df['null']]

In [7]:
merge_columns = ['groups', 'self', 'sample_delta']

In [8]:
value_columns = ['Hxy']

#### First turn after the current turn

In [9]:
# make a copy of our df
df_ = df.copy()

# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [10]:
# merge the copied doc with the actual doc using dyad ID, self,
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

(23233133, 14)


Hxy_x                    0
nx                       0
ny                       0
turn_delta               0
self                     0
x_speaker                0
dyad                     0
corpus                   0
conversation_id          0
x_turn_id                0
null                     0
groups                   0
sample_delta             0
Hxy_y              1093574
dtype: int64

#### Second turn after the current turn

In [11]:
# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [12]:
# merge the copied doc with the actual doc using dyad ID, self,
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

(23233133, 15)


Hxy_x                    0
nx                       0
ny                       0
turn_delta               0
self                     0
x_speaker                0
dyad                     0
corpus                   0
conversation_id          0
x_turn_id                0
null                     0
groups                   0
sample_delta             0
Hxy_y              1093574
Hxy                2171042
dtype: int64

#### Final checks

In [13]:
# check what row x column combos have NaNs
df.isna().sum()

Hxy_x                    0
nx                       0
ny                       0
turn_delta               0
self                     0
x_speaker                0
dyad                     0
corpus                   0
conversation_id          0
x_turn_id                0
null                     0
groups                   0
sample_delta             0
Hxy_y              1093574
Hxy                2171042
dtype: int64

In [14]:
# get a sample of the NaN containing rows
df[['groups', 'sample_delta', 'Hxy_x', 'Hxy_y', 'Hxy']].loc[df['Hxy'].isna()].sample(n=10).head(n=10)

,groups,sample_delta,Hxy_x,Hxy_y,Hxy
2869387,callhome-4522-60,30,3.721467,NaN,NaN
19473921,se1-ag3-se1-ag3-se1_ag3_m_02_1-412,28,6.197931,5.766410,NaN
14127983,MICASE-office_hours-MICASE-office_hours-ofc280...,15,4.062446,4.293159,NaN
2932873,callhome-4574-387,21,3.209975,2.987523,NaN
97714,GCSAusE-GCSAusE-30-90,9,1.230796,NaN,NaN
1278452,SBCSAE-SBCSAE-51-500,22,5.727911,5.664495,NaN
2817228,callhome-4431-212,30,2.610899,NaN,NaN
6568972,CABNC-KBC-CABNC-KBC-KBCRE009-280,23,6.255895,NaN,NaN
12564235,MICASE-lel-MICASE-lel-lel305ju092-305,31,1.596344,NaN,NaN
14585578,MICASE-seminars-MICASE-seminars-sem475mx041-575,25,16.938503,NaN,NaN


In [15]:
df = df.loc[~df['Hxy'].isna()]
df.shape

(21062091, 15)

In [16]:
df = df.rename(columns={'resid': 'residual'})
df = df.rename(columns={'Hxy_x': 'resid', 'Hxy_y': 'resid_1', 'Hxy': 'resid_2'})

In [17]:
df[['groups', 'resid', 'resid_1', 'resid_2']].head()

,groups,resid,resid_1,resid_2
0,GCSAusE-GCSAusE-01-6,6.344819,6.000488,5.521488
1,GCSAusE-GCSAusE-01-6,6.000488,5.521488,6.057978
2,GCSAusE-GCSAusE-01-6,5.521488,6.057978,6.198739
3,GCSAusE-GCSAusE-01-6,6.057978,6.198739,5.745489
4,GCSAusE-GCSAusE-01-6,6.198739,5.745489,5.604022


#### Create Allan Variance Scale Parameter

In [18]:
group_cols = ['groups', 'self']
number_cols = ['sample_delta']

In [19]:
N = df[group_cols+number_cols].groupby(by=group_cols).agg('count').reset_index()
N = N.rename(columns={'sample_delta': 'N'})

In [20]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=N, right_on=group_cols,
    how='left'
)
df.shape

(21062091, 16)

In [21]:
tau = df[group_cols+number_cols].groupby(by=group_cols).agg('max').reset_index()
tau = tau.rename(columns={'sample_delta': 'tau'})

In [22]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=tau, right_on=group_cols,
    how='left'
)
df.shape

(21062091, 17)

In [23]:
df.head()

,resid,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta,resid_1,resid_2,N,tau
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,1,6.000488,5.521488,28,28
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,2,5.521488,6.057978,28,28
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,3,6.057978,6.198739,28,28
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,4,6.198739,5.745489,28,28
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,5,5.745489,5.604022,28,28


#### Calculate Allan $\sigma^2$

In [24]:
df = df.loc[~df['resid_2'].isna()]

In [25]:
df['allan_var'] = (1/(2*(df['tau']**2))) * ((df['resid_2'] - (2*df['resid_1']) + df['resid'])**2)

In [26]:
df.head()

,resid,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta,resid_1,resid_2,N,tau,allan_var
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,1,6.000488,5.521488,28,28,0.000012
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,2,5.521488,6.057978,28,28,0.000658
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,3,6.057978,6.198739,28,28,0.000100
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,4,6.198739,5.745489,28,28,0.000225
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-6,5,5.745489,5.604022,28,28,0.000062


In [27]:
df['null'] = False

#### checkpoint

In [28]:
df.to_parquet('regular_data.parquet')

### Null data

In [29]:
df = main_df.loc[main_df['null']]

In [30]:
merge_columns = ['groups', 'self', 'sample_delta']

In [31]:
value_columns = ['Hxy']

#### First turn after the current turn

In [32]:
# make a copy of our df
df_ = df.copy()

# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [33]:
# merge the copied doc with the actual doc using dyad ID, self,
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

(2191693, 14)


Hxy_x                   0
nx                      0
ny                      0
turn_delta              0
self                    0
x_speaker               0
dyad                    0
corpus                  0
conversation_id         0
x_turn_id               0
null                    0
groups                  0
sample_delta            0
Hxy_y              101185
dtype: int64

#### Second turn after the current turn

In [34]:
# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [35]:
# merge the copied doc with the actual doc using dyad ID, self,
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

(2191693, 15)


Hxy_x                   0
nx                      0
ny                      0
turn_delta              0
self                    0
x_speaker               0
dyad                    0
corpus                  0
conversation_id         0
x_turn_id               0
null                    0
groups                  0
sample_delta            0
Hxy_y              101185
Hxy                202348
dtype: int64

#### Final checks

In [36]:
# check what row x column combos have NaNs
df.isna().sum()

Hxy_x                   0
nx                      0
ny                      0
turn_delta              0
self                    0
x_speaker               0
dyad                    0
corpus                  0
conversation_id         0
x_turn_id               0
null                    0
groups                  0
sample_delta            0
Hxy_y              101185
Hxy                202348
dtype: int64

In [37]:
# get a sample of the NaN containing rows
df[['groups', 'sample_delta', 'Hxy_x', 'Hxy_y', 'Hxy']].loc[df['Hxy'].isna()].sample(n=10).head(n=10)

,groups,sample_delta,Hxy_x,Hxy_y,Hxy
1072446,null-MICASE-advising-MICASE-advising-adv700ju0...,24,7.360013,NaN,NaN
1778678,null-se1-ag1-se1-ag1-se1_ag1_m_03_1-527,24,2.065610,NaN,NaN
472583,null-CABNC-0missing-CABNC-0missing-KCGRE006-73,20,12.770847,NaN,NaN
1859394,null-se1-ag3-se1-ag3-se1_ag3_m_02_2-154,26,2.071614,NaN,NaN
694613,null-CABNC-KBH-CABNC-KBH-KBHRE00F-461,20,1.396105,NaN,NaN
1672046,null-se0-ag3-se0-ag3-se0_ag3_f_03_1-832,21,1.741669,1.740673,NaN
1204997,null-MICASE-lel-MICASE-lel-lel320ju147-73,6,6.153245,NaN,NaN
1716791,null-se0-ag4-se0-ag4-se0_ag4_f_02_1-263,44,2.894483,NaN,NaN
345425,null-SBCSAE-SBCSAE-48-598,22,1.310872,NaN,NaN
1888372,null-se1-ag4-se1-ag4-se1_ag4_m_02_1-312,20,3.096557,NaN,NaN


In [38]:
df = df.loc[~df['Hxy'].isna()]
df.shape

(1989345, 15)

In [39]:
df = df.rename(columns={'resid': 'residual'})
df = df.rename(columns={'Hxy_x': 'resid', 'Hxy_y': 'resid_1', 'Hxy': 'resid_2'})

In [40]:
df[['groups', 'resid', 'resid_1', 'resid_2']].head()

,groups,resid,resid_1,resid_2
0,null-callfriend-eng_n-4175-19,8.396281,9.874767,8.867210
1,null-callfriend-eng_n-4175-19,9.874767,8.867210,9.359125
2,null-callfriend-eng_n-4175-19,8.867210,9.359125,8.732200
3,null-callfriend-eng_n-4175-19,9.359125,8.732200,9.067421
4,null-callfriend-eng_n-4175-19,8.732200,9.067421,8.825766


#### Create Allan Variance Scale Parameter

In [41]:
group_cols = ['groups', 'self']
number_cols = ['sample_delta']

In [42]:
N = df[group_cols+number_cols].groupby(by=group_cols).agg('count').reset_index()
N = N.rename(columns={'sample_delta': 'N'})

In [43]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=N, right_on=group_cols,
    how='left'
)
df.shape

(1989345, 16)

In [44]:
tau = df[group_cols+number_cols].groupby(by=group_cols).agg('max').reset_index()
tau = tau.rename(columns={'sample_delta': 'tau'})

In [45]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=tau, right_on=group_cols,
    how='left'
)
df.shape

(1989345, 17)

In [46]:
df.head()

,resid,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta,resid_1,resid_2,N,tau
0,8.396281,29.0,25.0,1,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,1,9.874767,8.867210,17,17
1,9.874767,29.0,5.0,3,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,2,8.867210,9.359125,17,17
2,8.867210,29.0,16.0,6,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,3,9.359125,8.732200,17,17
3,9.359125,29.0,5.0,8,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,4,8.732200,9.067421,17,17
4,8.732200,29.0,8.0,11,False,callfriend-eng_n-4175-M1,callfriend-eng_n-4175-M1-callfriend-eng_n-4175-M2,callfriend-eng_n,callfriend-eng_n-4175,null-callfriend-eng_n-4175-19,True,null-callfriend-eng_n-4175-19,5,9.067421,8.825766,17,17


#### Calculate Allan $\sigma^2$

In [47]:
df = df.loc[~df['resid_2'].isna()]

In [48]:
df['allan_var'] = (1/(2*(df['tau']**2))) * ((df['resid_2'] - (2*df['resid_1']) + df['resid'])**2)

In [49]:
df[['groups', 'resid', 'resid_1', 'resid_2', 'allan_var']].head(n=20)

,groups,resid,resid_1,resid_2,allan_var
0,null-callfriend-eng_n-4175-19,8.396281,9.874767,8.867210,0.010693
1,null-callfriend-eng_n-4175-19,9.874767,8.867210,9.359125,0.003890
2,null-callfriend-eng_n-4175-19,8.867210,9.359125,8.732200,0.002166
3,null-callfriend-eng_n-4175-19,9.359125,8.732200,9.067421,0.001602
4,null-callfriend-eng_n-4175-19,8.732200,9.067421,8.825766,0.000576
5,null-callfriend-eng_n-4175-19,9.067421,8.825766,9.799860,0.002557
6,null-callfriend-eng_n-4175-19,8.825766,9.799860,9.738371,0.001855
7,null-callfriend-eng_n-4175-19,9.799860,9.738371,8.778589,0.001396
8,null-callfriend-eng_n-4175-19,9.738371,8.778589,8.320789,0.000436
9,null-callfriend-eng_n-4175-19,8.778589,8.320789,8.584559,0.000901


In [50]:
df['null'] = True

#### checkpoint

In [51]:
df.to_parquet('null_data.parquet')

# Save outputs

In [4]:
df = [pd.read_parquet('null_data.parquet'), pd.read_parquet('regular_data.parquet')]

#### Quick check that data is actually different.

In [7]:
df[0][['groups', 'resid', 'resid_1', 'resid_2', 'allan_var']].head(n=20)

,groups,resid,resid_1,resid_2,allan_var
0,null-callfriend-eng_n-4175-19,8.396281,9.874767,8.867210,0.010693
1,null-callfriend-eng_n-4175-19,9.874767,8.867210,9.359125,0.003890
2,null-callfriend-eng_n-4175-19,8.867210,9.359125,8.732200,0.002166
3,null-callfriend-eng_n-4175-19,9.359125,8.732200,9.067421,0.001602
4,null-callfriend-eng_n-4175-19,8.732200,9.067421,8.825766,0.000576
5,null-callfriend-eng_n-4175-19,9.067421,8.825766,9.799860,0.002557
6,null-callfriend-eng_n-4175-19,8.825766,9.799860,9.738371,0.001855
7,null-callfriend-eng_n-4175-19,9.799860,9.738371,8.778589,0.001396
8,null-callfriend-eng_n-4175-19,9.738371,8.778589,8.320789,0.000436
9,null-callfriend-eng_n-4175-19,8.778589,8.320789,8.584559,0.000901


In [8]:
check_tids = [tid.replace('null-', '') for tid in df[0]['groups'].unique()[:10]]
df[1][['groups', 'resid', 'resid_1', 'resid_2', 'allan_var']].loc[df[1]['groups'].isin(check_tids)].head(n=20)

,groups,resid,resid_1,resid_2,allan_var
1655892,callfriend-eng_n-4175-19,9.724691,9.718081,9.729723,3.149186e-07
1655893,callfriend-eng_n-4175-19,9.718081,9.729723,7.685726,3.994001e-03
1655894,callfriend-eng_n-4175-19,9.729723,7.685726,8.528924,7.878916e-03
1655895,callfriend-eng_n-4175-19,7.685726,8.528924,9.681519,9.047859e-05
1655896,callfriend-eng_n-4175-19,8.528924,9.681519,8.492232,5.183748e-03
1655897,callfriend-eng_n-4175-19,9.681519,8.492232,9.530680,4.690734e-03
1655898,callfriend-eng_n-4175-19,8.492232,9.530680,8.980171,2.386371e-03
1655899,callfriend-eng_n-4175-19,9.530680,8.980171,8.417046,1.504646e-07
1655900,callfriend-eng_n-4175-19,8.980171,8.417046,8.962903,1.162423e-03
1655901,callfriend-eng_n-4175-19,8.417046,8.962903,8.727501,5.769059e-04


#### Save data to final .parquet

In [9]:
df = pd.concat(df, ignore_index=True)

In [10]:
# df.to_csv(
#     allan_var,
#     index=False,
#     encoding='utf-8',
#     sep='\t'
# )
df.to_parquet(allan_var)

#### Clean up data

In [11]:
os.remove('null_data.parquet')
os.remove('regular_data.parquet')